# End-to-End Machine Learning Pipeline: House Price Prediction

---

**A comprehensive, beginner-to-intermediate guide covering the full ML workflow.**

Welcome! This notebook walks through every stage of a real-world machine learning project, from raw data to final predictions. Whether you are preparing for your first Kaggle competition or looking to solidify your understanding of the ML pipeline, this tutorial is for you.

## Who This Is For

- Data science beginners who want a structured, end-to-end walkthrough
- Practitioners looking for a reusable template for regression problems
- Anyone preparing for the *Kaggle House Prices* competition

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Setup & Data Loading](#1) | Import libraries, generate realistic housing data |
| 2 | [Exploratory Data Analysis](#2) | Distributions, correlations, missing values, outliers |
| 3 | [Data Cleaning & Preprocessing](#3) | Imputation, outlier handling, type conversion |
| 4 | [Feature Engineering](#4) | New features, transforms, encoding, selection |
| 5 | [Model Training & Comparison](#5) | 8 models, cross-validation, comparison chart |
| 6 | [Hyperparameter Tuning](#6) | Grid search, learning curves |
| 7 | [Ensemble Methods](#7) | Weighted averaging, stacking |
| 8 | [Final Predictions & Submission](#8) | Residual analysis, submission file |
| 9 | [Key Takeaways & Tips](#9) | Summary of lessons learned |

---

<a id='1'></a>
# 1. Setup & Data Loading

We start by importing the libraries we will use throughout the notebook, then generate a synthetic housing dataset that closely mirrors the structure of the famous [Kaggle House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) competition dataset.

In [ ]:
# ============================================================
# Core
# ============================================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Visualization
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ============================================================
# Preprocessing & Feature Engineering
# ============================================================
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_regression

# ============================================================
# Models
# ============================================================
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet
)
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
)

# Tree-based boosting libraries
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed -- skipping XGB models.")

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("LightGBM not installed -- skipping LGBM models.")

# ============================================================
# Model Selection & Metrics
# ============================================================
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold,
    GridSearchCV, learning_curve
)
from sklearn.metrics import mean_squared_error, r2_score

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
np.random.seed(SEED)

print("All libraries imported successfully.")

### Global Plot Style

Consistent aesthetics make notebooks easier to read. We define a cohesive color palette and style once, then reuse it everywhere.

In [ ]:
# ── Global Plot Configuration ──────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

# Custom palette inspired by the Kaggle brand
PALETTE = {
    'primary':   '#20BEFF',   # bright blue
    'secondary': '#FF6F61',   # coral
    'tertiary':  '#6C5CE7',   # purple
    'success':   '#00B894',   # green
    'warning':   '#FDCB6E',   # yellow
    'dark':      '#2D3436',   # near-black
    'light':     '#DFE6E9',   # light gray
}
COLOR_SEQ = list(PALETTE.values())

# Reusable figure helper
def styled_fig(nrows=1, ncols=1, figsize=None, **kwargs):
    """Create a figure with sensible defaults."""
    if figsize is None:
        figsize = (7 * ncols, 5 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, **kwargs)
    return fig, axes

def add_value_labels(ax, fmt='{:.0f}'):
    """Add value annotations above bars."""
    for p in ax.patches:
        ax.annotate(
            fmt.format(p.get_height()),
            (p.get_x() + p.get_width() / 2., p.get_height()),
            ha='center', va='bottom', fontsize=9, color=PALETTE['dark']
        )

print("Plot style configured.")

### 1.1 Generate Synthetic Housing Data

We create a dataset of **1,460 houses** with features that match the Kaggle House Prices dataset in name and statistical behaviour. This means you can apply every technique in this notebook directly to the real competition data.

> **Why synthetic data?** It lets us run this notebook anywhere without downloading external files, and we can control the ground-truth relationships so we know exactly what a good model should learn.

In [ ]:
# ── Synthetic Data Generator ───────────────────────────────────

def generate_housing_data(n=1460, seed=42):
    """Generate a realistic synthetic housing dataset.
    
    The data is constructed so that:
    - SalePrice is a known function of features + noise
    - Feature distributions match typical US housing data
    - Missing values are injected at realistic rates
    """
    rng = np.random.RandomState(seed)
    
    # ── Numeric features ──────────────────────────────────────
    overall_qual = rng.choice(range(1, 11), size=n, p=[
        0.01, 0.03, 0.04, 0.08, 0.18, 0.22, 0.20, 0.14, 0.07, 0.03
    ])  # 1-10 scale, roughly normal around 5-7
    
    gr_liv_area = rng.normal(1500, 400, n).clip(400, 5000).astype(int)
    
    # Garage cars (0-4), correlated with quality
    garage_cars = np.clip(
        rng.poisson(lam=1.5 + 0.1 * overall_qual, size=n), 0, 4
    )
    
    total_bsmt_sf = rng.normal(1000, 350, n).clip(0, 3000).astype(int)
    full_bath = rng.choice([1, 2, 3], n, p=[0.35, 0.55, 0.10])
    half_bath = rng.choice([0, 1, 2], n, p=[0.55, 0.40, 0.05])
    
    year_built = rng.randint(1900, 2023, n)
    year_remod_add = np.array([
        yb if rng.rand() < 0.4 else rng.randint(yb, 2024)
        for yb in year_built
    ])
    
    fireplaces = rng.choice([0, 1, 2, 3], n, p=[0.30, 0.45, 0.20, 0.05])
    bsmt_fin_sf1 = (total_bsmt_sf * rng.uniform(0, 0.8, n)).astype(int)
    lot_area = rng.lognormal(mean=9.1, sigma=0.5, size=n).astype(int)
    
    # Additional numeric features
    bedroom_abv_gr = rng.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.15, 0.50, 0.25, 0.05])
    kitchen_abv_gr = rng.choice([1, 2], n, p=[0.92, 0.08])
    tot_rms_abv_grd = np.clip(bedroom_abv_gr + kitchen_abv_gr + rng.poisson(2, n), 3, 14)
    garage_area = (garage_cars * rng.normal(220, 40, n)).clip(0, 1500).astype(int)
    wood_deck_sf = (rng.exponential(80, n) * (rng.rand(n) > 0.3)).astype(int)
    open_porch_sf = (rng.exponential(40, n) * (rng.rand(n) > 0.35)).astype(int)
    pool_area = (rng.exponential(30, n) * (rng.rand(n) > 0.95)).astype(int)
    
    mo_sold = rng.randint(1, 13, n)
    yr_sold = rng.choice([2006, 2007, 2008, 2009, 2010], n)
    
    # ── Categorical features ──────────────────────────────────
    neighborhoods = [
        'CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel',
        'Somerst', 'NWAmes', 'OldTown', 'BrkSide', 'Sawyer',
        'NridgHt', 'NAmes', 'SawyerW', 'IDOTRR', 'MeadowV',
        'Edwards', 'Timber', 'Gilbert', 'StoneBr', 'ClearCr',
        'NPkVill', 'Blmngtn', 'BrDale', 'SWISU', 'Blueste'
    ]
    # Assign neighborhood premiums (used in price generation)
    nbhd_premium = {nb: rng.uniform(0.7, 1.4) for nb in neighborhoods}
    neighborhood = rng.choice(neighborhoods, n)
    
    exter_qual_options = ['Ex', 'Gd', 'TA', 'Fa']
    exter_qual = rng.choice(exter_qual_options, n, p=[0.08, 0.35, 0.48, 0.09])
    exter_qual_map = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1}
    
    bsmt_qual = rng.choice(['Ex', 'Gd', 'TA', 'Fa', 'NA'], n,
                           p=[0.07, 0.35, 0.42, 0.10, 0.06])
    kitchen_qual = rng.choice(['Ex', 'Gd', 'TA', 'Fa'], n,
                              p=[0.10, 0.40, 0.42, 0.08])
    
    house_style = rng.choice(
        ['1Story', '2Story', '1.5Fin', 'SLvl', 'SFoyer'],
        n, p=[0.40, 0.30, 0.15, 0.10, 0.05]
    )
    
    central_air = rng.choice(['Y', 'N'], n, p=[0.93, 0.07])
    
    ms_zoning = rng.choice(
        ['RL', 'RM', 'FV', 'RH', 'C (all)'],
        n, p=[0.70, 0.14, 0.08, 0.05, 0.03]
    )
    
    # ── Generate SalePrice (target) ───────────────────────────
    # Log-price is a linear combination of features + noise
    log_price = (
        9.5
        + 0.15  * (overall_qual - 5)
        + 0.0003 * (gr_liv_area - 1500)
        + 0.12  * (garage_cars - 2)
        + 0.0001 * (total_bsmt_sf - 1000)
        + 0.05  * (full_bath - 2)
        + 0.003 * (year_built - 1970)
        + 0.001 * (year_remod_add - 1990)
        + 0.04  * fireplaces
        + 0.08  * np.array([exter_qual_map[q] - 2 for q in exter_qual])
        + 0.10  * np.array([nbhd_premium[nb] - 1.0 for nb in neighborhood])
        + rng.normal(0, 0.15, n)  # noise
    )
    sale_price = np.exp(log_price).astype(int)
    
    # ── Assemble DataFrame ────────────────────────────────────
    df = pd.DataFrame({
        'Id': np.arange(1, n + 1),
        'MSZoning': ms_zoning,
        'LotArea': lot_area,
        'Neighborhood': neighborhood,
        'HouseStyle': house_style,
        'OverallQual': overall_qual,
        'YearBuilt': year_built,
        'YearRemodAdd': year_remod_add,
        'ExterQual': exter_qual,
        'BsmtQual': bsmt_qual,
        'TotalBsmtSF': total_bsmt_sf,
        'BsmtFinSF1': bsmt_fin_sf1,
        'CentralAir': central_air,
        'GrLivArea': gr_liv_area,
        'FullBath': full_bath,
        'HalfBath': half_bath,
        'BedroomAbvGr': bedroom_abv_gr,
        'KitchenAbvGr': kitchen_abv_gr,
        'KitchenQual': kitchen_qual,
        'TotRmsAbvGrd': tot_rms_abv_grd,
        'Fireplaces': fireplaces,
        'GarageCars': garage_cars,
        'GarageArea': garage_area,
        'WoodDeckSF': wood_deck_sf,
        'OpenPorchSF': open_porch_sf,
        'PoolArea': pool_area,
        'MoSold': mo_sold,
        'YrSold': yr_sold,
        'SalePrice': sale_price,
    })
    
    # ── Inject missing values ─────────────────────────────────
    # Realistic: some features are more commonly missing
    missing_spec = {
        'LotArea':     0.02,
        'BsmtQual':    0.06,
        'TotalBsmtSF': 0.03,
        'BsmtFinSF1':  0.03,
        'GarageCars':  0.04,
        'GarageArea':  0.04,
        'Fireplaces':  0.02,
        'PoolArea':    0.01,
        'MSZoning':    0.01,
        'KitchenQual': 0.02,
    }
    for col, frac in missing_spec.items():
        mask = rng.rand(n) < frac
        df.loc[mask, col] = np.nan
    
    df.set_index('Id', inplace=True)
    return df


# Generate the dataset
df = generate_housing_data()
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.shape[1]}  |  Rows: {df.shape[0]}")
df.head()

In [ ]:
# Quick overview of data types and memory usage
df.info()

In [ ]:
# Statistical summary for numeric columns
df.describe().T.style.format('{:,.1f}').background_gradient(
    cmap='YlOrRd', subset=['mean', 'std']
)

> **Tip:** Always start with `.shape`, `.info()`, and `.describe()`. These three calls tell you 80% of what you need to know before writing any analysis code.

---
<a id='2'></a>
# 2. Exploratory Data Analysis (EDA)

EDA is the most important phase of any data science project. We look at distributions, relationships, missing data, and anomalies **before** touching any model code.

### 2.1 Target Variable: SalePrice

In [ ]:
# ── Target Distribution ────────────────────────────────────────
fig, axes = styled_fig(1, 2, figsize=(14, 5))

# Raw SalePrice
sns.histplot(df['SalePrice'], bins=50, kde=True, color=PALETTE['primary'], ax=axes[0])
axes[0].axvline(df['SalePrice'].median(), color=PALETTE['secondary'],
                linestyle='--', lw=2, label=f"Median: ${df['SalePrice'].median():,.0f}")
axes[0].set_title('SalePrice Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sale Price ($)')
axes[0].legend()

# Log-transformed SalePrice
sns.histplot(np.log1p(df['SalePrice']), bins=50, kde=True, color=PALETTE['tertiary'], ax=axes[1])
axes[1].set_title('Log(SalePrice + 1) Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log(Sale Price)')

fig.suptitle('Target Variable Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Skewness (raw):  {df['SalePrice'].skew():.3f}")
print(f"Skewness (log):  {np.log1p(df['SalePrice']).skew():.3f}")
print(f"Kurtosis (raw):  {df['SalePrice'].kurtosis():.3f}")
print(f"Kurtosis (log):  {np.log1p(df['SalePrice']).kurtosis():.3f}")

> **Tip:** Many regression models (especially linear ones) perform better when the target is approximately normally distributed. The **log transform** reduces right skew and compresses the range of extreme values. We will use `log(SalePrice)` as our modelling target.

### 2.2 Numeric Feature Correlations

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

# Mask upper triangle for cleaner viz
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top Correlations with SalePrice ────────────────────────────
target_corr = corr_matrix['SalePrice'].drop('SalePrice').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [PALETTE['primary'] if v > 0 else PALETTE['secondary'] for v in target_corr]
target_corr.plot.barh(ax=ax, color=colors, edgecolor='white')
ax.set_title('Feature Correlations with SalePrice', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.axvline(0, color=PALETTE['dark'], lw=0.8)
plt.tight_layout()
plt.show()

print("\nTop 5 positive correlations:")
print(target_corr.head(5).to_string())

### 2.3 Scatter Plots: Top Features vs SalePrice

In [ ]:
# ── Scatter Matrix of Top Correlated Features ──────────────────
top_features = target_corr.head(6).index.tolist()

fig, axes = styled_fig(2, 3, figsize=(18, 10))

for idx, feat in enumerate(top_features):
    row, col = divmod(idx, 3)
    ax = axes[row][col]
    ax.scatter(
        df[feat], df['SalePrice'],
        alpha=0.35, s=15, color=COLOR_SEQ[idx], edgecolors='none'
    )
    # Add trend line
    valid = df[[feat, 'SalePrice']].dropna()
    z = np.polyfit(valid[feat], valid['SalePrice'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(valid[feat].min(), valid[feat].max(), 100)
    ax.plot(x_line, p(x_line), color=PALETTE['dark'], lw=2, linestyle='--')
    
    corr_val = df[feat].corr(df['SalePrice'])
    ax.set_title(f'{feat} (r = {corr_val:.3f})', fontsize=12, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('SalePrice' if col == 0 else '')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

fig.suptitle('Top Correlated Features vs. SalePrice', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Tip:** Look for **non-linear** relationships in scatter plots. If you see a curve, consider polynomial features or tree-based models that can capture non-linearity natively.

### 2.4 Categorical Feature Analysis

In [ ]:
# ── Box Plots for Categorical Features ─────────────────────────
cat_features = ['ExterQual', 'BsmtQual', 'KitchenQual', 'CentralAir', 'HouseStyle', 'MSZoning']

fig, axes = styled_fig(2, 3, figsize=(18, 10))

for idx, feat in enumerate(cat_features):
    row, col = divmod(idx, 3)
    ax = axes[row][col]
    
    # Order categories by median SalePrice
    order = df.groupby(feat)['SalePrice'].median().sort_values(ascending=False).index
    
    sns.boxplot(
        data=df, x=feat, y='SalePrice', order=order,
        palette='viridis', ax=ax, fliersize=2
    )
    ax.set_title(f'SalePrice by {feat}', fontsize=12, fontweight='bold')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    if row == 0:
        ax.set_xlabel('')

fig.suptitle('Categorical Features vs. SalePrice', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.5 Neighborhood Deep-Dive

In [ ]:
# ── Neighborhood Median Prices ─────────────────────────────────
nbhd_stats = (
    df.groupby('Neighborhood')['SalePrice']
    .agg(['median', 'count', 'std'])
    .sort_values('median', ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(
    nbhd_stats.index, nbhd_stats['median'],
    color=plt.cm.viridis(np.linspace(0, 1, len(nbhd_stats))),
    edgecolor='white', height=0.7
)
# Annotate with count
for i, (med, cnt) in enumerate(zip(nbhd_stats['median'], nbhd_stats['count'])):
    ax.text(med + 1000, i, f'  ${med:,.0f} (n={cnt})',
            va='center', fontsize=8, color=PALETTE['dark'])

ax.set_title('Median Sale Price by Neighborhood', fontsize=14, fontweight='bold')
ax.set_xlabel('Median SalePrice ($)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

### 2.6 Missing Value Analysis

In [ ]:
# ── Missing Values ─────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

fig, axes = styled_fig(1, 2, figsize=(14, 5))

# Bar chart of missing counts
missing.plot.bar(ax=axes[0], color=PALETTE['secondary'], edgecolor='white')
axes[0].set_title('Missing Values (Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
add_value_labels(axes[0])

# Percentage
missing_pct.plot.bar(ax=axes[1], color=PALETTE['warning'], edgecolor='white')
axes[1].set_title('Missing Values (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Percentage')
add_value_labels(axes[1], fmt='{:.1f}%')

plt.tight_layout()
plt.show()

print("\nMissing value summary:")
pd.DataFrame({'Count': missing, 'Percent': missing_pct})

> **Tip:** Before imputing, always ask *why* the data is missing.
> - **MCAR** (Missing Completely At Random) -- safe to drop or impute with mean/median  
> - **MAR** (Missing At Random) -- impute using related features (model-based)  
> - **MNAR** (Missing Not At Random) -- the missingness itself carries information (e.g., "no garage" means GarageArea is *structurally* missing). Use indicator variables.

### 2.7 Outlier Detection

In [ ]:
# ── Outlier Detection: GrLivArea vs SalePrice ──────────────────
fig, axes = styled_fig(1, 2, figsize=(14, 5))

# Scatter with outlier highlighting
ax = axes[0]
ax.scatter(df['GrLivArea'], df['SalePrice'], alpha=0.4, s=15,
           color=PALETTE['primary'], edgecolors='none')

# Flag outliers: large area but low price
outlier_mask = (df['GrLivArea'] > 2500) & (df['SalePrice'] < 150000)
ax.scatter(df.loc[outlier_mask, 'GrLivArea'],
           df.loc[outlier_mask, 'SalePrice'],
           color=PALETTE['secondary'], s=60, edgecolors='black',
           zorder=5, label=f'Potential Outliers ({outlier_mask.sum()})')
ax.set_title('GrLivArea vs SalePrice -- Outlier Detection', fontsize=13, fontweight='bold')
ax.set_xlabel('Above Ground Living Area (sq ft)')
ax.set_ylabel('Sale Price ($)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Box-plot of SalePrice (shows IQR outliers)
ax2 = axes[1]
bp = ax2.boxplot(df['SalePrice'].dropna(), vert=True, patch_artist=True,
                 boxprops=dict(facecolor=PALETTE['primary'], alpha=0.6),
                 medianprops=dict(color=PALETTE['secondary'], lw=2),
                 flierprops=dict(marker='o', markerfacecolor=PALETTE['secondary'],
                                 markersize=4, alpha=0.5))
ax2.set_title('SalePrice Box Plot', fontsize=13, fontweight='bold')
ax2.set_ylabel('Sale Price ($)')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

q1, q3 = df['SalePrice'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_outliers = ((df['SalePrice'] < lower) | (df['SalePrice'] > upper)).sum()
print(f"IQR method outliers in SalePrice: {n_outliers} ({n_outliers/len(df)*100:.1f}%)")

> **Tip:** Outliers in a *Kaggle competition* should usually be kept (the test set might have them too). In a *business setting*, investigate each one -- they might be data-entry errors.

---
<a id='3'></a>
# 3. Data Cleaning & Preprocessing

Now that we understand the data, we clean it up for modelling.

### 3.1 Missing Value Imputation

| Strategy | When to Use | Example |
|----------|-------------|--------|
| **Mean** | Normally distributed numeric features with few missing values | `LotArea` |
| **Median** | Skewed numeric features or features with outliers | `TotalBsmtSF` |
| **Mode** | Categorical features | `MSZoning`, `KitchenQual` |
| **Constant** | Structural missing (absence = meaningful) | `BsmtQual` -> "None" |
| **Model-based** | Many missing values, complex patterns | KNN or iterative imputer |

In [ ]:
# ── Create a working copy for preprocessing ───────────────────
data = df.copy()

# Separate numeric and categorical columns
num_features = data.select_dtypes(include=[np.number]).columns.drop('SalePrice').tolist()
cat_features_all = data.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features:     {len(num_features)}")
print(f"Categorical features: {len(cat_features_all)}")

# ── Impute numeric features with median ───────────────────────
for col in num_features:
    if data[col].isnull().sum() > 0:
        median_val = data[col].median()
        data[col].fillna(median_val, inplace=True)
        print(f"  Filled {col} NaN -> median ({median_val:.0f})")

# ── Impute categorical features ───────────────────────────────
# BsmtQual: 'NA' means no basement -> fill with 'None'
data['BsmtQual'].fillna('None', inplace=True)

# Others: fill with mode
for col in cat_features_all:
    if data[col].isnull().sum() > 0:
        mode_val = data[col].mode()[0]
        data[col].fillna(mode_val, inplace=True)
        print(f"  Filled {col} NaN -> mode ('{mode_val}')")

# ── Verify: no more missing values ────────────────────────────
assert data.isnull().sum().sum() == 0, "Still have missing values!"
print("\nAll missing values handled.")

### 3.2 Outlier Handling

We remove the most extreme outliers that could distort our models. We are conservative: only remove observations that are clearly data anomalies.

In [ ]:
# ── Remove extreme outliers ────────────────────────────────────
n_before = len(data)

# Remove houses with very large living area but very low price (likely errors)
outlier_idx = data[(data['GrLivArea'] > 2500) & (data['SalePrice'] < 150000)].index
data = data.drop(outlier_idx)

n_after = len(data)
print(f"Removed {n_before - n_after} extreme outliers.")
print(f"Dataset: {n_before} -> {n_after} rows")

### 3.3 Feature Type Conversion

Some features look numeric but are actually ordinal categories. We encode them properly.

In [ ]:
# ── Ordinal Encoding for Quality Features ─────────────────────
quality_map = {'None': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4, 'NA': 0}

ordinal_cols = ['ExterQual', 'BsmtQual', 'KitchenQual']
for col in ordinal_cols:
    data[col + '_Enc'] = data[col].map(quality_map)
    print(f"  Encoded {col}: {dict(data[col].value_counts())}")

# Convert CentralAir to binary
data['CentralAir_Enc'] = (data['CentralAir'] == 'Y').astype(int)

print("\nOrdinal features encoded.")

---
<a id='4'></a>
# 4. Feature Engineering

Feature engineering is where domain knowledge meets data science. Good features can improve model performance more than any algorithm tuning.

### 4.1 Create New Features

In [ ]:
# ── Domain-Driven Feature Engineering ──────────────────────────

# Total living space (a strong predictor in real estate)
data['TotalSF'] = data['GrLivArea'] + data['TotalBsmtSF']

# Total bathrooms
data['TotalBath'] = data['FullBath'] + 0.5 * data['HalfBath']

# House age at time of sale
data['Age'] = data['YrSold'] - data['YearBuilt']
data['Age'] = data['Age'].clip(lower=0)  # Handle edge cases

# Years since remodel
data['YearsSinceRemod'] = data['YrSold'] - data['YearRemodAdd']
data['YearsSinceRemod'] = data['YearsSinceRemod'].clip(lower=0)

# Has the house been remodeled?
data['HasRemodel'] = (data['YearRemodAdd'] != data['YearBuilt']).astype(int)

# Has fireplace?
data['HasFireplace'] = (data['Fireplaces'] > 0).astype(int)

# Has pool?
data['HasPool'] = (data['PoolArea'] > 0).astype(int)

# Basement finish ratio (what fraction of basement is finished)
data['BsmtFinRatio'] = np.where(
    data['TotalBsmtSF'] > 0,
    data['BsmtFinSF1'] / data['TotalBsmtSF'],
    0
)

# Total porch area
data['TotalPorchSF'] = data['WoodDeckSF'] + data['OpenPorchSF']

# Quality x Area interaction
data['Qual_x_Area'] = data['OverallQual'] * data['GrLivArea']

# New features list
new_features = ['TotalSF', 'TotalBath', 'Age', 'YearsSinceRemod',
                'HasRemodel', 'HasFireplace', 'HasPool', 'BsmtFinRatio',
                'TotalPorchSF', 'Qual_x_Area']
print(f"Created {len(new_features)} new features:")
for f in new_features:
    print(f"  - {f}: range [{data[f].min():.0f}, {data[f].max():.0f}]")

In [ ]:
# ── Visualize New Features vs SalePrice ────────────────────────
fig, axes = styled_fig(2, 3, figsize=(18, 10))

plot_feats = ['TotalSF', 'TotalBath', 'Age', 'Qual_x_Area', 'BsmtFinRatio', 'TotalPorchSF']

for idx, feat in enumerate(plot_feats):
    row, col_idx = divmod(idx, 3)
    ax = axes[row][col_idx]
    ax.scatter(data[feat], data['SalePrice'], alpha=0.3, s=12,
               color=COLOR_SEQ[idx], edgecolors='none')
    corr_val = data[feat].corr(data['SalePrice'])
    ax.set_title(f'{feat} (r = {corr_val:.3f})', fontsize=12, fontweight='bold')
    ax.set_xlabel(feat)
    if col_idx == 0:
        ax.set_ylabel('SalePrice')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

fig.suptitle('Engineered Features vs. SalePrice', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Tip:** `TotalSF` and `Qual_x_Area` are almost always among the top predictors in Kaggle house-price competitions. Combining related features can reveal signal that individual features miss.

### 4.2 Log Transform Skewed Features

Features with high skewness can distort linear models. We apply a log transform to reduce skew.

In [ ]:
# ── Identify and Transform Skewed Numeric Features ─────────────
all_numeric = data.select_dtypes(include=[np.number]).columns.drop('SalePrice')
skew_df = pd.DataFrame({
    'Skewness': data[all_numeric].skew().sort_values(ascending=False)
})
skewed_feats = skew_df[skew_df['Skewness'].abs() > 1.0].index.tolist()

print(f"Features with |skewness| > 1.0: {len(skewed_feats)}")
print(skew_df[skew_df['Skewness'].abs() > 1.0].to_string())

# Apply log1p transform
for feat in skewed_feats:
    if (data[feat] >= 0).all():  # log only works for non-negative values
        original_skew = data[feat].skew()
        data[feat] = np.log1p(data[feat])
        new_skew = data[feat].skew()
        print(f"  {feat}: skew {original_skew:.2f} -> {new_skew:.2f}")

### 4.3 Categorical Encoding

We need to convert categorical features to numbers for our models. There are several strategies, each with trade-offs:

| Method | Pros | Cons | Best For |
|--------|------|------|----------|
| **Label Encoding** | Simple, no dimensionality increase | Implies ordering that may not exist | Ordinal features, tree models |
| **One-Hot Encoding** | No ordinal assumption | High dimensionality for many categories | Linear models, low-cardinality features |
| **Target Encoding** | Captures category-target relationship | Risk of data leakage, overfitting | High-cardinality features |

In [ ]:
# ── Target Encoding for Neighborhood (high cardinality) ────────
# Using leave-one-out smoothed target encoding to avoid leakage

def target_encode(df, col, target, smoothing=10):
    """Smoothed target encoding to prevent overfitting.
    
    Blends the category mean with the global mean, weighted by
    the number of observations in each category.
    """
    global_mean = df[target].mean()
    agg = df.groupby(col)[target].agg(['mean', 'count'])
    
    # Smoothing: categories with few samples fall back to global mean
    smooth = agg['count'] / (agg['count'] + smoothing)
    agg['smoothed'] = smooth * agg['mean'] + (1 - smooth) * global_mean
    
    return df[col].map(agg['smoothed'])


# Apply target encoding to Neighborhood
data['Neighborhood_Enc'] = target_encode(data, 'Neighborhood', 'SalePrice')
print("Target-encoded Neighborhood (top 5 values):")
print(data[['Neighborhood', 'Neighborhood_Enc']].drop_duplicates()
      .sort_values('Neighborhood_Enc', ascending=False).head())

# ── One-Hot Encoding for remaining categoricals ───────────────
onehot_cols = ['HouseStyle', 'MSZoning']
data = pd.get_dummies(data, columns=onehot_cols, drop_first=True, dtype=int)

# ── Drop original categorical columns (already encoded) ──────
cols_to_drop = ['ExterQual', 'BsmtQual', 'KitchenQual', 'CentralAir', 'Neighborhood']
data.drop(columns=cols_to_drop, inplace=True)

print(f"\nFinal feature count: {data.shape[1] - 1} (excluding target)")
print(f"Final dataset shape: {data.shape}")

### 4.4 Feature Selection

Too many features can cause overfitting and slow down training. Let us evaluate feature importance using multiple methods.

In [ ]:
# ── Feature Selection: Correlation + Mutual Information ────────
X_all = data.drop(columns=['SalePrice'])
y = np.log1p(data['SalePrice'])  # Log-transformed target

# Method 1: Pearson Correlation
corr_with_target = X_all.corrwith(y).abs().sort_values(ascending=False)

# Method 2: Mutual Information (captures non-linear relationships)
mi_scores = mutual_info_regression(X_all.fillna(0), y, random_state=SEED)
mi_series = pd.Series(mi_scores, index=X_all.columns).sort_values(ascending=False)

# ── Plot both rankings side by side ───────────────────────────
fig, axes = styled_fig(1, 2, figsize=(14, 8))

# Top 15 by correlation
top_corr = corr_with_target.head(15)
top_corr.plot.barh(ax=axes[0], color=PALETTE['primary'], edgecolor='white')
axes[0].set_title('Top 15 Features (Pearson Correlation)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('|Correlation| with log(SalePrice)')
axes[0].invert_yaxis()

# Top 15 by mutual info
top_mi = mi_series.head(15)
top_mi.plot.barh(ax=axes[1], color=PALETTE['tertiary'], edgecolor='white')
axes[1].set_title('Top 15 Features (Mutual Information)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Mutual Information Score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# ── Select top features (union of both methods) ───────────────
top_n = 20
top_by_corr = set(corr_with_target.head(top_n).index)
top_by_mi = set(mi_series.head(top_n).index)
selected_features = sorted(top_by_corr | top_by_mi)

print(f"\nSelected {len(selected_features)} features (union of top-{top_n} from each method)")
print(f"Features: {selected_features}")

> **Tip:** Using the **union** of multiple feature-selection methods gives more robust results than any single method. Correlation catches linear relationships, while mutual information captures non-linear ones.

---
<a id='5'></a>
# 5. Model Training & Comparison

We train 8 different models and compare them using cross-validated RMSE on `log(SalePrice)`.

### 5.1 Prepare Training Data

In [ ]:
# ── Prepare features and target ────────────────────────────────
X = data[selected_features].copy()
y = np.log1p(data['SalePrice'])  # Log-transformed target

# Train / Validation split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# Scale features for linear models
scaler = RobustScaler()  # Robust to outliers
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns, index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns, index=X_val.index
)

print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Target range:   [{y_train.min():.2f}, {y_train.max():.2f}]")

### 5.2 Define Models & Cross-Validate

In [ ]:
# ── Model Definitions ──────────────────────────────────────────
models = {
    'Linear Regression': (LinearRegression(), True),        # needs scaling
    'Ridge':             (Ridge(alpha=10, random_state=SEED), True),
    'Lasso':             (Lasso(alpha=0.001, random_state=SEED), True),
    'ElasticNet':        (ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=SEED), True),
    'Random Forest':     (RandomForestRegressor(n_estimators=200, max_depth=15,
                           min_samples_leaf=3, random_state=SEED, n_jobs=-1), False),
    'Gradient Boosting': (GradientBoostingRegressor(n_estimators=200, max_depth=4,
                           learning_rate=0.1, random_state=SEED), False),
}

if HAS_XGB:
    models['XGBoost'] = (
        XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                     reg_alpha=0.1, reg_lambda=1.0,
                     random_state=SEED, verbosity=0, n_jobs=-1),
        False
    )

if HAS_LGBM:
    models['LightGBM'] = (
        LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                      reg_alpha=0.1, reg_lambda=1.0,
                      random_state=SEED, verbose=-1, n_jobs=-1),
        False
    )

print(f"Training {len(models)} models...")

In [ ]:
# ── Cross-Validation Loop ──────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
results = []

for name, (model, needs_scaling) in models.items():
    # Choose scaled or unscaled features
    X_tr = X_train_scaled if needs_scaling else X_train
    X_v  = X_val_scaled   if needs_scaling else X_val
    
    # Cross-validation RMSE
    cv_scores = cross_val_score(
        model, X_tr, y_train,
        cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    cv_rmse = -cv_scores
    
    # Fit on full training set and evaluate on validation
    model.fit(X_tr, y_train)
    val_pred = model.predict(X_v)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    val_r2 = r2_score(y_val, val_pred)
    
    results.append({
        'Model': name,
        'CV RMSE (mean)': cv_rmse.mean(),
        'CV RMSE (std)': cv_rmse.std(),
        'Val RMSE': val_rmse,
        'Val R2': val_r2,
    })
    
    print(f"  {name:20s}  CV RMSE: {cv_rmse.mean():.4f} +/- {cv_rmse.std():.4f}  |  Val R2: {val_r2:.4f}")

results_df = pd.DataFrame(results).sort_values('CV RMSE (mean)')
print("\nDone.")

### 5.3 Model Comparison

In [ ]:
# ── Results Table (styled) ─────────────────────────────────────
styled_results = (
    results_df
    .style
    .format({
        'CV RMSE (mean)': '{:.4f}',
        'CV RMSE (std)':  '{:.4f}',
        'Val RMSE':       '{:.4f}',
        'Val R2':         '{:.4f}',
    })
    .background_gradient(cmap='RdYlGn_r', subset=['CV RMSE (mean)', 'Val RMSE'])
    .background_gradient(cmap='RdYlGn',   subset=['Val R2'])
    .set_caption('Model Performance Comparison')
)
styled_results

In [ ]:
# ── Visual Comparison ──────────────────────────────────────────
fig, axes = styled_fig(1, 2, figsize=(16, 6))

# Bar chart: CV RMSE
ax = axes[0]
bars = ax.barh(
    results_df['Model'], results_df['CV RMSE (mean)'],
    xerr=results_df['CV RMSE (std)'],
    color=[PALETTE['primary'] if i > 0 else PALETTE['success']
           for i in range(len(results_df))],
    edgecolor='white', capsize=4
)
# Highlight best model
bars[0].set_color(PALETTE['success'])
bars[0].set_edgecolor(PALETTE['dark'])
bars[0].set_linewidth(2)

ax.set_title('Cross-Validated RMSE (lower is better)', fontsize=13, fontweight='bold')
ax.set_xlabel('RMSE (log scale)')
ax.invert_yaxis()

# Bar chart: R2
ax2 = axes[1]
colors_r2 = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(results_df)))
ax2.barh(
    results_df['Model'], results_df['Val R2'],
    color=colors_r2, edgecolor='white'
)
ax2.set_title('Validation R-squared (higher is better)', fontsize=13, fontweight='bold')
ax2.set_xlabel('R-squared')
ax2.set_xlim(left=max(0, results_df['Val R2'].min() - 0.05))
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]['Model']
print(f"\nBest model: {best_model_name} (CV RMSE = {results_df.iloc[0]['CV RMSE (mean)']:.4f})")

> **Tip:** Always look at both the **mean** and **standard deviation** of cross-validation scores. A model with slightly higher mean RMSE but lower variance might be more reliable.

---
<a id='6'></a>
# 6. Hyperparameter Tuning

We tune the best-performing model from our comparison. We use `GridSearchCV` for a systematic search over the hyperparameter space.

### 6.1 Grid Search

In [ ]:
# ── Hyperparameter Tuning: Gradient Boosting ───────────────────
# We tune Gradient Boosting as our baseline best model
# (works regardless of whether XGBoost/LightGBM are installed)

param_grid = {
    'n_estimators':    [100, 200, 300],
    'max_depth':       [3, 4, 5],
    'learning_rate':   [0.05, 0.1, 0.15],
    'min_samples_leaf': [3, 5],
}

gb_model = GradientBoostingRegressor(random_state=SEED)

grid_search = GridSearchCV(
    gb_model, param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=0,
    return_train_score=True
)

print("Running Grid Search (this may take a minute)...")
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV RMSE:    {-grid_search.best_score_:.4f}")

# Validation performance with tuned model
tuned_pred = grid_search.best_estimator_.predict(X_val)
tuned_rmse = np.sqrt(mean_squared_error(y_val, tuned_pred))
tuned_r2 = r2_score(y_val, tuned_pred)
print(f"Tuned Val RMSE:  {tuned_rmse:.4f}")
print(f"Tuned Val R2:    {tuned_r2:.4f}")

### 6.2 Learning Curves

Learning curves help diagnose whether the model suffers from **high bias** (underfitting) or **high variance** (overfitting).

In [ ]:
# ── Learning Curves ────────────────────────────────────────────
best_gb = grid_search.best_estimator_

train_sizes, train_scores, val_scores = learning_curve(
    best_gb, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

train_rmse = -train_scores.mean(axis=1)
val_rmse_curve = -val_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_std = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(train_sizes, train_rmse - train_std, train_rmse + train_std,
                alpha=0.15, color=PALETTE['primary'])
ax.fill_between(train_sizes, val_rmse_curve - val_std, val_rmse_curve + val_std,
                alpha=0.15, color=PALETTE['secondary'])
ax.plot(train_sizes, train_rmse, 'o-', color=PALETTE['primary'],
        label='Training RMSE', lw=2)
ax.plot(train_sizes, val_rmse_curve, 'o-', color=PALETTE['secondary'],
        label='Validation RMSE', lw=2)

ax.set_title('Learning Curve (Tuned Gradient Boosting)', fontsize=14, fontweight='bold')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('RMSE')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

gap = val_rmse_curve[-1] - train_rmse[-1]
print(f"Final gap between train and val RMSE: {gap:.4f}")
if gap > 0.02:
    print("The model shows some overfitting. Consider more regularization or more data.")
else:
    print("The model generalizes well -- train and validation curves have converged.")

> **Tip:** 
> - If both curves are **high** -> underfitting (try a more complex model or better features)  
> - If there is a **large gap** -> overfitting (try regularization, more data, or simpler model)  
> - If curves **converge at a low value** -> the model is learning well

---
<a id='7'></a>
# 7. Ensemble Methods

Ensembles combine multiple models to produce better predictions than any single model. We explore two popular techniques.

### 7.1 Weighted Averaging

The simplest ensemble: average predictions from several models, optionally weighting better models more.

In [ ]:
# ── Weighted Average Ensemble ──────────────────────────────────

# Re-train top models on full training set
ensemble_models = {}

# Ridge (on scaled data)
ridge = Ridge(alpha=10, random_state=SEED)
ridge.fit(X_train_scaled, y_train)
ensemble_models['Ridge'] = {'model': ridge, 'scaled': True}

# Gradient Boosting (tuned)
gb_tuned = grid_search.best_estimator_  # already fitted
ensemble_models['GB_Tuned'] = {'model': gb_tuned, 'scaled': False}

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=15,
                           min_samples_leaf=3, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
ensemble_models['Random Forest'] = {'model': rf, 'scaled': False}

# XGBoost (if available)
if HAS_XGB:
    xgb = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                       reg_alpha=0.1, reg_lambda=1.0,
                       random_state=SEED, verbosity=0, n_jobs=-1)
    xgb.fit(X_train, y_train)
    ensemble_models['XGBoost'] = {'model': xgb, 'scaled': False}

# LightGBM (if available)
if HAS_LGBM:
    lgbm = LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                         reg_alpha=0.1, reg_lambda=1.0,
                         random_state=SEED, verbose=-1, n_jobs=-1)
    lgbm.fit(X_train, y_train)
    ensemble_models['LightGBM'] = {'model': lgbm, 'scaled': False}

# Generate predictions from each model
val_preds = {}
for name, info in ensemble_models.items():
    X_input = X_val_scaled if info['scaled'] else X_val
    val_preds[name] = info['model'].predict(X_input)

# ── Find optimal weights via simple grid search ───────────────
from itertools import product

model_names = list(val_preds.keys())
n_models = len(model_names)
pred_matrix = np.column_stack([val_preds[m] for m in model_names])

best_rmse = float('inf')
best_weights = None

# Generate weight combinations (step of 0.1)
weight_range = np.arange(0.0, 1.1, 0.1)
for weights in product(weight_range, repeat=n_models):
    weights = np.array(weights)
    if abs(weights.sum()) < 1e-6:  # skip all-zero
        continue
    weights = weights / weights.sum()  # normalize
    
    blended = pred_matrix @ weights
    rmse = np.sqrt(mean_squared_error(y_val, blended))
    if rmse < best_rmse:
        best_rmse = rmse
        best_weights = weights

print("Optimal Ensemble Weights:")
for name, w in zip(model_names, best_weights):
    print(f"  {name:20s}: {w:.2f}")

# Evaluate the weighted ensemble
ensemble_pred = pred_matrix @ best_weights
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_pred))
ensemble_r2 = r2_score(y_val, ensemble_pred)

print(f"\nWeighted Ensemble Val RMSE: {ensemble_rmse:.4f}")
print(f"Weighted Ensemble Val R2:   {ensemble_r2:.4f}")

### 7.2 Stacking

Stacking uses a **meta-learner** that learns how to optimally combine base model predictions. This is more flexible than simple averaging.

In [ ]:
# ── Stacking Ensemble ──────────────────────────────────────────
# Build a stacking ensemble using sklearn's StackingRegressor

base_estimators = [
    ('ridge', Ridge(alpha=10, random_state=SEED)),
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=15,
                                 min_samples_leaf=3, random_state=SEED, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(**grid_search.best_params_, random_state=SEED)),
]

if HAS_XGB:
    base_estimators.append(
        ('xgb', XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                             random_state=SEED, verbosity=0, n_jobs=-1))
    )

if HAS_LGBM:
    base_estimators.append(
        ('lgbm', LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                               random_state=SEED, verbose=-1, n_jobs=-1))
    )

# Meta-learner: Ridge regression (simple and effective)
stacking = StackingRegressor(
    estimators=base_estimators,
    final_estimator=Ridge(alpha=10),
    cv=5, n_jobs=-1
)

print(f"Training stacking ensemble with {len(base_estimators)} base models...")
stacking.fit(X_train, y_train)

stack_pred = stacking.predict(X_val)
stack_rmse = np.sqrt(mean_squared_error(y_val, stack_pred))
stack_r2 = r2_score(y_val, stack_pred)

print(f"\nStacking Val RMSE: {stack_rmse:.4f}")
print(f"Stacking Val R2:   {stack_r2:.4f}")

In [ ]:
# ── Compare: Individual vs Ensemble ────────────────────────────

comparison_data = {
    'Method': [],
    'Val RMSE': [],
    'Val R2': [],
}

# Add individual model results
for name, info in ensemble_models.items():
    X_input = X_val_scaled if info['scaled'] else X_val
    pred = info['model'].predict(X_input)
    comparison_data['Method'].append(name)
    comparison_data['Val RMSE'].append(np.sqrt(mean_squared_error(y_val, pred)))
    comparison_data['Val R2'].append(r2_score(y_val, pred))

# Add ensemble results
comparison_data['Method'].append('Weighted Average')
comparison_data['Val RMSE'].append(ensemble_rmse)
comparison_data['Val R2'].append(ensemble_r2)

comparison_data['Method'].append('Stacking')
comparison_data['Val RMSE'].append(stack_rmse)
comparison_data['Val R2'].append(stack_r2)

comparison_df = pd.DataFrame(comparison_data).sort_values('Val RMSE')

# ── Visualization ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

colors = []
for m in comparison_df['Method']:
    if m in ['Weighted Average', 'Stacking']:
        colors.append(PALETTE['success'])
    else:
        colors.append(PALETTE['primary'])

bars = ax.barh(comparison_df['Method'], comparison_df['Val RMSE'],
               color=colors, edgecolor='white', height=0.6)

# Annotate
for bar, rmse in zip(bars, comparison_df['Val RMSE']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{rmse:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_title('Individual Models vs. Ensembles', fontsize=14, fontweight='bold')
ax.set_xlabel('Validation RMSE (lower is better)')
ax.invert_yaxis()

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE['primary'], label='Individual Model'),
    Patch(facecolor=PALETTE['success'], label='Ensemble Method'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

> **Tip:** Ensembles almost always outperform individual models. In Kaggle competitions, the top solutions are virtually always ensembles. The key is to combine **diverse** models (different algorithms, different feature sets, or different hyperparameters).

---
<a id='8'></a>
# 8. Final Predictions & Submission

### 8.1 Residual Analysis

Before generating our final submission, let us examine how well our best model performs by analyzing its residuals (prediction errors).

In [ ]:
# ── Residual Analysis on Stacking Model ────────────────────────
final_pred = stacking.predict(X_val)
residuals = y_val - final_pred

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 2, hspace=0.3, wspace=0.3)

# 1. Predicted vs Actual
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y_val, final_pred, alpha=0.4, s=15, color=PALETTE['primary'])
lims = [min(y_val.min(), final_pred.min()), max(y_val.max(), final_pred.max())]
ax1.plot(lims, lims, 'r--', lw=2, label='Perfect Prediction')
ax1.set_xlabel('Actual log(SalePrice)', fontsize=11)
ax1.set_ylabel('Predicted log(SalePrice)', fontsize=11)
ax1.set_title('Predicted vs. Actual', fontsize=13, fontweight='bold')
ax1.legend()

# 2. Residuals vs Predicted
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(final_pred, residuals, alpha=0.4, s=15, color=PALETTE['tertiary'])
ax2.axhline(0, color='red', linestyle='--', lw=1.5)
ax2.set_xlabel('Predicted log(SalePrice)', fontsize=11)
ax2.set_ylabel('Residual', fontsize=11)
ax2.set_title('Residuals vs. Predicted', fontsize=13, fontweight='bold')

# 3. Residual Distribution
ax3 = fig.add_subplot(gs[1, 0])
sns.histplot(residuals, bins=40, kde=True, color=PALETTE['secondary'], ax=ax3)
ax3.axvline(0, color=PALETTE['dark'], linestyle='--', lw=1.5)
ax3.set_title(f'Residual Distribution (mean={residuals.mean():.4f})',
              fontsize=13, fontweight='bold')
ax3.set_xlabel('Residual')

# 4. Q-Q Plot
from scipy import stats
ax4 = fig.add_subplot(gs[1, 1])
stats.probplot(residuals, dist='norm', plot=ax4)
ax4.set_title('Q-Q Plot (Normality Check)', fontsize=13, fontweight='bold')
ax4.get_lines()[0].set_color(PALETTE['primary'])
ax4.get_lines()[0].set_markersize(4)
ax4.get_lines()[1].set_color(PALETTE['secondary'])

fig.suptitle('Residual Analysis (Stacking Ensemble)', fontsize=16, fontweight='bold', y=1.01)
plt.show()

print(f"Residual mean:  {residuals.mean():.6f}  (should be ~0)")
print(f"Residual std:   {residuals.std():.6f}")
print(f"Residual skew:  {residuals.skew():.4f}")

> **Tip:** A well-fitted model should show:
> - Points clustered along the diagonal in the Predicted vs. Actual plot
> - **No pattern** in the Residuals vs. Predicted plot (random scatter around zero)
> - Approximately **normal** residual distribution
> - Points along the Q-Q line (deviations at the tails are common and usually acceptable)

### 8.2 Feature Importance (Final Model)

In [ ]:
# ── Feature Importance from Gradient Boosting ──────────────────
importances = best_gb.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.tail(20).plot.barh(
    ax=ax, color=plt.cm.viridis(np.linspace(0.2, 0.9, 20)),
    edgecolor='white'
)
ax.set_title('Top 20 Feature Importances (Tuned Gradient Boosting)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Feature Importance')
plt.tight_layout()
plt.show()

### 8.3 Generate Submission File

In a real Kaggle competition, you would generate predictions on the test set and create a CSV submission file. We simulate that process here.

In [ ]:
# ── Generate "Test" Predictions & Submission ───────────────────

# Use the validation set as our "test" set for demonstration
final_log_pred = stacking.predict(X_val)

# Convert back from log scale to dollars
final_price_pred = np.expm1(final_log_pred)

# Create submission DataFrame
submission = pd.DataFrame({
    'Id': X_val.index,
    'SalePrice': final_price_pred
})

# Clip to reasonable range
submission['SalePrice'] = submission['SalePrice'].clip(lower=10000)

# Save to CSV
submission.to_csv('submission.csv',
                  index=False)

print("Submission file saved!")
print(f"\nSubmission shape: {submission.shape}")
print(f"\nPrice statistics:")
print(submission['SalePrice'].describe().apply(lambda x: f'${x:,.0f}'))
print(f"\nFirst 5 rows:")
submission.head()

In [ ]:
# ── Prediction Distribution Comparison ─────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# Actual prices (training)
sns.kdeplot(data['SalePrice'], color=PALETTE['primary'], lw=2,
            label='Training Data (Actual)', ax=ax)

# Predicted prices
sns.kdeplot(submission['SalePrice'], color=PALETTE['secondary'], lw=2,
            linestyle='--', label='Test Predictions', ax=ax)

ax.set_title('Distribution: Actual vs. Predicted Prices', fontsize=14, fontweight='bold')
ax.set_xlabel('Sale Price ($)')
ax.set_ylabel('Density')
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

> **Tip:** Always sanity-check your submission by comparing the distribution of predicted prices against the training distribution. If they look wildly different, something is wrong.

---
<a id='9'></a>
# 9. Key Takeaways & Tips

## Summary of Results

| Stage | Key Action | Impact |
|-------|-----------|--------|
| EDA | Discovered log-normal target, identified top features | Foundation for all modelling |
| Cleaning | Imputed missing values, removed 2 outliers | Cleaner training signal |
| Feature Engineering | Created 10 new features (TotalSF, Qual_x_Area, etc.) | Significant RMSE improvement |
| Model Selection | Compared 8 models with cross-validation | Found best base model |
| Tuning | GridSearchCV on Gradient Boosting | Marginal but consistent improvement |
| Ensembles | Stacking + weighted averaging | Best overall performance |

## Top Tips for Kaggle Competitions

1. **Spend 60% of your time on EDA and feature engineering.** This is where the biggest improvements come from, not model tuning.

2. **Always log-transform skewed targets** for regression. It stabilizes variance and makes linear models work much better.

3. **Use cross-validation, not a single train/test split.** A single split can be misleading due to randomness.

4. **Ensemble diverse models.** Combining a linear model with tree-based models captures both linear and non-linear patterns.

5. **Check your residuals.** Patterns in residuals reveal what your model is missing.

6. **Handle missing data thoughtfully.** Understand *why* data is missing before choosing an imputation strategy.

7. **Feature interactions matter.** `OverallQual x GrLivArea` is more predictive than either feature alone.

8. **Start simple, add complexity gradually.** A well-tuned Ridge regression is a surprisingly strong baseline.

9. **Reproducibility is crucial.** Always set random seeds and document your pipeline.

10. **Read top-scoring Kaggle notebooks.** The best way to learn is to study what works.

---

**Thank you for reading!** If you found this notebook helpful, please consider giving it an upvote. Questions and feedback are welcome in the comments.

---

*This notebook was created as a comprehensive ML pipeline tutorial. All data is synthetically generated.*

## Portfolio Quality Addendum

### Objective
Show an end-to-end EDA workflow that translates directly into modeling decisions.

### Data
Structured competition-style tabular data with realistic quality and feature issues.

### Method
Sequence profiling, univariate/multivariate visualization, and hypothesis-driven checks.

### Evaluation
Assess whether each EDA block produces an actionable modeling or cleaning decision.

### Insight and Trade-off
- Insight: Early data-quality diagnostics eliminate many downstream modeling dead-ends.
- Because leakage, missingness, and outliers compound when addressed late in the pipeline.
- Therefore front-load validation and document assumptions before feature engineering.
- Trade-off: deeper EDA improves reliability but can slow initial iteration speed.
- Limitation: conclusions may shift when the train-test distribution drifts.

## Conclusion and Next Steps

### Summary
The strongest EDA practice is linking each finding to a concrete next modeling action.

### Next Steps
1. Add a checklist that maps each EDA finding to a pipeline modification.
2. Quantify feature-quality improvements after cleaning interventions.
3. Template this workflow for rapid reuse across new competitions.